### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="physiochemical_protein",
    dataset_year="2013",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5QW3H",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/physiochemical_protein/ && wget -P local-data-warehouse/physiochemical_protein/ https://archive.ics.uci.edu/static/public/265/physicochemical+properties+of+protein+tertiary+structure.zip && unzip local-data-warehouse/physiochemical_protein/physicochemical+properties+of+protein+tertiary+structure.zip -d local-data-warehouse/physiochemical_protein/
""",
    # References
    academic_reference_bibtex=r"""@misc{rana2013protein,
  author       = {Rana, Prashant},
  title        = {Physicochemical Properties of Protein Tertiary Structure},
  year         = {2013},
  howpublished = {\url{https://doi.org/10.24432/C5QW3H}},
  note         = {UCI Machine Learning Repository},
}
""",
    academic_reference_bibtex_key="rana2013protein",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We renamed the features to be more semantically meaningful.
- We log1p scale the target.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Log1pResidualSize",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import  numpy as np
df = pd.read_csv(f"{dataset_mold.path}/CASP.csv")

target_feature = "Log1pResidualSize"
df.columns = [
    target_feature,
    "TotalSurfaceArea",
    "NonPolarExposedArea",
    "FracExposedNonPolarResidue",
    "FracExposedNonPolarPart",
    "MassWeightedExposedArea",
    "AvgDeviationExposedArea",
    "EuclideanDistance",
    "SecondaryStructurePenalty",
    "SpatialDistNK",
]
df[target_feature] = np.log1p(df[target_feature])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 45,730
Columns: 10
Use sampling: False (sample size: 45,730)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['MassWeightedExposedArea', 'FracExposedNonPolarPart', 'TotalSurfaceArea', 'NonPolarExposedArea', 'EuclideanDistance', 'AvgDeviationExposedArea', 'SpatialDistNK', 'FracExposedNonPolarResidue', 'SecondaryStructurePenalty']
Rows remaining as candidates after top-9 filter: 3,240 (of 45,730)

#### Duplicate Report
Total duplicate rows: 1711 (3.74% of dataset)
Duplicate rows ignoring target: 1756 (3.84% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Log1pResidualSize,TotalSurfaceArea,NonPolarExposedArea,FracExposedNonPolarResidue,FracExposedNonPolarPart,MassWeightedExposedArea,AvgDeviationExposedArea,EuclideanDistance,SecondaryStructurePenalty,SpatialDistNK
0,2.988607,7031.44,2456.97,0.34942,47.0180,1.008540e+06,88.0866,3998.06,3,38.7263
1,1.041689,17099.10,5994.03,0.35054,223.8060,2.316828e+06,278.8630,5468.55,135,23.3121
2,1.316944,16079.50,5807.81,0.36119,180.9630,2.250300e+06,272.1780,5617.59,149,23.8631
3,1.674664,13600.60,3520.54,0.25885,167.1890,1.865227e+06,221.0140,4804.69,115,28.5555
4,1.394263,7876.41,1984.92,0.25200,88.5033,1.051353e+06,112.9370,3535.51,29,36.0818


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Log1pResidualSize,float64,0.0,0.0,15903.0,"0.0, 1.1006, 1.025, 1.0647, 1.1168, 1.0335, 1.0943, 1.0633, 1.0661, 1.0757"
1,TotalSurfaceArea,float64,0.0,0.0,39916.0,"13475.4, 5811.82, 4000.26, 14170.5, 20734.4, 4670.89, 15024.1, 4465.62, 14958.5, 4074.87"
2,NonPolarExposedArea,float64,0.0,0.0,39863.0,"4814.93, 1087.13, 1053.23, 2129.8, 1866.32, 7997.71, 3520.75, 3102.87, 1729.67, 2067.07"
3,FracExposedNonPolarResidue,float64,0.0,0.0,20089.0,"0.3573, 0.2718, 0.2485, 0.3118, 0.2663, 0.2807, 0.2873, 0.1812, 0.269, 0.2795"
4,FracExposedNonPolarPart,float64,0.0,0.0,40374.0,"168.55, 52.5591, 33.732, 186.407, 189.396, 46.7282, 174.306, 108.529, 49.8648, 56.5611"
5,MassWeightedExposedArea,float64,0.0,0.0,41868.0,"1877843.5474, 799977.1539, 569494.3892, 1937925.8075, 2876946.082, 686984.9356, 554911.2539, 952969.4869, 1258537.1269, 659783.0916"
6,AvgDeviationExposedArea,float64,0.0,0.0,39155.0,"227.605, 65.2332, 46.039, 66.6405, 356.061, 214.666, 118.232, 211.148, 132.03, 60.1525"
7,EuclideanDistance,float64,0.0,0.0,39450.0,"4644.75, 4057.08, 1773.46, 3034.98, 1399.62, 4628.02, 2334.29, 4581.39, 3903.68, 3193.36"
8,SpatialDistNK,float64,0.0,0.0,37299.0,"46.5464, 29.7563, 39.7659, 34.8833, 44.4892, 38.8321, 44.4197, 38.1176, 43.1816, 44.2712"
9,SecondaryStructurePenalty,int64,0.0,0.0,341.0,"32, 17, 40, 36, 30, 41, 33, 27, 39, 38"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Log1pResidualSize,45730.0,1.891191e+00,0.773012,0.0000,3.090997e+00
TotalSurfaceArea,45730.0,9.871597e+03,4058.138034,2392.0500,4.003490e+04
NonPolarExposedArea,45730.0,3.017367e+03,1464.324663,403.5000,1.531200e+04
FracExposedNonPolarResidue,45730.0,3.023919e-01,0.062886,0.0925,5.776900e-01
FracExposedNonPolarPart,45730.0,1.034924e+02,55.424985,10.3101,3.693170e+02
MassWeightedExposedArea,45730.0,1.368299e+06,564036.688407,319490.2166,5.472011e+06
AvgDeviationExposedArea,45730.0,1.456381e+02,69.999230,31.9704,5.984080e+02
EuclideanDistance,45730.0,3.989756e+03,1993.574575,0.0000,1.059482e+05
SecondaryStructurePenalty,45730.0,6.997507e+01,56.493443,0.0000,3.500000e+02
SpatialDistNK,45730.0,3.452366e+01,5.979755,15.2280,5.530090e+01


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.59,-0.046,-0.462,0.598,0.081,log1p,151240.9,3.781272e+14,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to physiochemical_protein/019d5d4e-a083-7edb-861a-a115682c9e57
019d5d4e-a083-7edb-861a-a115682c9e57
4c53ced2033f7901ab5da5e0a0e54453c918d429096c73baf87eb850b4e8b9cd
